## Question 1

### (a)
Trend is a scalar (-1,0,1). If the sum of trends i.e. Sv is same then phi(x) would also sum up to give the same result. E.g. if we have 3 neighbors with trends 0, 1, 0 and another node with 3 neighbors with trends -1, 1, 1. In both cases sum Sv would be 1 and phi(x) considering w=1, p=1 and c=1 would be:
for v1 (0,1,0) -> 1+2+1 = 4
for v2 (-1,1,1) -> 0+2+2 = 4

so the output is same when the number of neighbors Nv and sum of their trends Sv is the same. We can plug any value of p, c and w and the outputs would be same. 
Generic values would be:
$$ v1 (-1,1,1): p(c+w+c+c) = p(w+3c) $$
$$ v2 (-1,1,1): p(-w+c+w+c+w+c) = p(w+3c) $$

Thus, whatever values of w, p and c we plug in, result for both nodes will be same.

## (b)
Each node can have at most K neighbors such that 1<=Nv<=K where K is a known positive integer. To represent message in terms of K, we need a base that can retain the counts of different trends even after sum so they can be recovered. For this, if we assume K ranges from 0-9 then we would need base 10 to represent each trend as a separate place value of the sum. If we consider range 0-99, the base needed would be 100 and similarly, if the range is from 0-K (our situation where max number of neighbors is K) then base needed would be K+1. Now that our base is determined, we can represent the message value of each type of trend as:

$$ ϕ(0) = (K+1)^0 $$

$$ ϕ(1) = (K+1)^1 $$

$$ ϕ(-1) = (K+1)^2 $$

then Sum would be:

$$
S_v = \sum_{t \in \{-1,0,1\}} n_t \, \phi(t) \;=\; n_0 \cdot 1 \;+\; n_{+1}\,(K+1) \;+\; n_{-1}\,(K+1)^2
$$

For recovery, we will use the following rule:

$$
n_t = \left\lfloor \frac{M_v}{(K+1)^{p(t)}} \right\rfloor \bmod (K+1),
\qquad p(0)=0,\; p(+1)=1,\; p(-1)=2
$$

In [1]:
#Numpy Verification
import numpy as np
from itertools import product

K = 8
B = K + 1                      # base

# message per trend (one value per neighbour)
phi = {0: B**0, 1: B**1, -1: B**2}

# all triples (n_neg, n_zero, n_pos) with 1 <= total <= K
triples = np.array([t for t in product(range(K + 1), repeat=3)
                    if 1 <= sum(t) <= K])
n_neg, n_zero, n_pos = triples.T

# aggregated message: sum over trends of count * phi(trend)
M = n_neg * phi[-1] + n_zero * phi[0] + n_pos * phi[1]

# decoding rules 
dec_zero = M % B
dec_pos  = (M // B) % B
dec_neg  = (M // B**2) % B

# 1) decoding recovers every triple
assert np.array_equal(dec_neg,  n_neg)
assert np.array_equal(dec_zero, n_zero)
assert np.array_equal(dec_pos,  n_pos)

# 2) uniqueness: no two triples share a sum
assert len(np.unique(M)) == len(triples)

print(f"All {len(triples)} triples passed; max sum = {M.max()}")

All 164 triples passed; max sum = 648


###  (i) Count steady neighbours

**Sum**
Possible: Yes
Reason: Because counts of all trends can be recovered. first coordinate gives count of -1, second gives count of 0 and third gives count of 1. So we can easily get count of 0 from second coordinate. For example, for neighborhood (0,0,1,-1), aggregated vector is (1,2,1). Second coordinate gives us 2 which is the count of steady terms.

**Mean**
Possible: No
Reason: This gives contribution of each trend not the count and since total number of neighbors is not given so counts cannot be recovered. Considering 2 neighbors: (0, 0) and (0,0, 0) where both give aggregated vector: (0,1,0) even though counts of steady trend neighbors for both is different ( 2 vs 3).

**Max**
Possible: No
Reason: Counts cant be recovered because max only returns the max at each coordinate. Consider two neighborhoods: (0,0,-1,1,1) and (0,1,-1). Both have the same aggregated vector (1,1,1) even though the number of steady terms are different for both.

###  (ii) Decide whether more than half of the neighbours are slowing
**Sum**
Possible: Yes
Reason: Because counts can be recovered so we can see if more than half neighbors belong to slowing trend i.e. -1. For this, we just recover the count of slowing trend which is at first coordinate. We can divide this count by sum of all coordinates (essentially N_v) and if we get > 0.5 then more than half neighbors are slowing.

**Mean**
Possible: yes
Reason: Because contributions can be recovered. Get contribution of -1 from first index and if its > 0.5 then more than half neighbors are slowing

**Max**
Possible: No
Reason: Because counts/contributions cant be recovered. e.g. consider two neighborhoods: (-1,-1,-1,0,1) and (-1,0,1). Both give aggregated vector: (1,1,1) even though in first case, more than half the neighbors are slowing and in second case, that's not true.

###  (iii) Decide whether any neighbour is slowing
**Max**
Possible: Yes
Reason: Counts are recoverable so we can get count of slowing trend neighbors from first coordinate. If count > 0 then there are neighbors that are slowing.

**Max**
Possible: Yes
Reason: Contributions recoverable so if contribution of slowing trend neighbors i.e. value of first coordinate is > 0 then there are slowing neighbors.

**Max**
Possible: Yes 
Reason:  If the value of first coordinate = 1 then there is atleast one slowing neighbor in the neighborhood
